In [ ]:
# Install the SPICE library
!pip install -q spiceypy

# Download four of the five NASA kernel files (-q = quiet)
!wget -q https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls
!wget -q https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de440s.bsp
!wget -q https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00011.tpc
!wget -q https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/moon_pa_de440_200625.bpc

# The Moon frame file is in the a_old_versions folder (this link works)
!wget -q https://naif.jpl.nasa.gov/pub/naif/generic_kernels/fk/satellites/a_old_versions/moon_de440_220930.tf

# Check that all five files arrived
!ls -lh naif0012.tls de440s.bsp pck00011.tpc moon_pa_de440_200625.bpc moon_de440_220930.tf

In [ ]:
import os

# List of expected files
files = [
    "naif0012.tls",
    "de440s.bsp",
    "pck00011.tpc",
    "moon_pa_de440_200625.bpc",
    "moon_de440_220930.tf"
]

print("Verifying downloaded files:\n")
for f in files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    print(f"{f:<30} | Status: {'[FOUND]' if exists else '[MISSING]':<10} | Size: {size} bytes")

In [ ]:
import numpy as np
import spiceypy as spice
import matplotlib.pyplot as plt

# Clear any cached or partially loaded kernels
spice.kclear()

for k in ["naif0012.tls", "de440s.bsp", "pck00011.tpc",
          "moon_de440_220930.tf", "moon_pa_de440_200625.bpc"]:
    spice.furnsh(k)

FRAME = "MOON_ME_DE440_ME421"   # Moon-fixed frame (mean Earth / rotation axis)
R = 1737.4                      # Moon radius in km

def elevation(body, et, lat_deg, lon_deg):
    lat, lon = np.radians(lat_deg), np.radians(lon_deg)
    up = np.array([np.cos(lat)*np.cos(lon),
                   np.cos(lat)*np.sin(lon),
                   np.sin(lat)])                 # "straight up" at the site
    site = R * up                                # site position from Moon's center
    pos, _ = spice.spkpos(body, et, FRAME, "NONE", "MOON")
    v = np.array(pos) - site                     # vector from the site to the body
    v = v / np.linalg.norm(v)
    return np.degrees(np.arcsin(v @ up))         # angle above the horizon

In [ ]:
et0 = spice.str2et("2026-11-14 00:00:00")        # string date -> ephemeris time
ets = et0 + np.arange(0, 30*86400, 3600)         # every hour for 30 days
days = (ets - et0) / 86400

lat, lon = -89.9, 0                              # a demo point near the south pole
sun = np.array([elevation("SUN", t, lat, lon) for t in ets])
earth = np.array([elevation("EARTH", t, lat, lon) for t in ets])

plt.plot(days, sun, label="Sun")
plt.plot(days, earth, label="Earth")
plt.axhline(0, color="gray")
plt.xlabel("Days"); plt.ylabel("Elevation (deg)")
plt.legend(); plt.grid(True); plt.show()

print("Sun above horizon:  ", (sun > 0).mean()*100, "% of the time")
print("Earth above horizon:", (earth > 0).mean()*100, "% of the time")

In [ ]:
import pandas as pd

sites = {
    "Site A": (-89.9, 0),
    "Site B": (-88.0, 30),
    "Site C": (-85.0, 120),
}

rows = []
for name, (la, lo) in sites.items():
    s = np.array([elevation("SUN", t, la, lo) for t in ets])
    e = np.array([elevation("EARTH", t, la, lo) for t in ets])
    rows.append({
        "site": name,
        "sun_percent": round((s > 0).mean()*100, 1),
        "earth_percent": round((e > 0).mean()*100, 1),
        "max_sun_elev": round(s.max(), 2),
    })

df = pd.DataFrame(rows)
df

In [ ]:
sites = {
    "Apollo 11":      (0.67,   23.47),
    "Apollo 12":      (-3.01, -23.42),
    "Apollo 14":      (-3.65, -17.47),
    "Apollo 15":      (26.13,   3.63),
    "Apollo 16":      (-8.97,  15.50),
    "Apollo 17":      (20.19,  30.77),
    "Chandrayaan-3":  (-69.37, 32.32),
    "Chang'e 4":      (-45.46, 177.59),
    "Near S. pole":   (-89.9,   0),
}

In [ ]:
rows = []
for name, (la, lo) in sites.items():
    s = np.array([elevation("SUN", t, la, lo) for t in ets])
    e = np.array([elevation("EARTH", t, la, lo) for t in ets])
    rows.append({
        "site": name,
        "sun_percent": round((s > 0).mean()*100, 1),
        "earth_percent": round((e > 0).mean()*100, 1),
        "max_sun_elev": round(s.max(), 1),
    })

pd.DataFrame(rows)

In [ ]:
!pip install -q astroquery

In [ ]:
from astroquery.jplhorizons import Horizons
from astropy.time import Time

lat, lon = -3.01, -23.42      # Apollo 12, as an example
when = "2026-11-14 12:00:00"
jd = Time(when, scale="utc").jd

loc = {"lon": lon, "lat": lat, "elevation": 0, "body": 301}   # 301 = the Moon

sun_h = Horizons(id="10",  location=loc, epochs=jd).ephemerides(quantities="4")
earth_h = Horizons(id="399", location=loc, epochs=jd).ephemerides(quantities="4")

print("Horizons  Sun elevation:  ", float(sun_h["EL"][0]))
print("Horizons  Earth elevation:", float(earth_h["EL"][0]))

et = spice.str2et(when)
print("Your code Sun elevation:  ", elevation("SUN", et, lat, lon))
print("Your code Earth elevation:", elevation("EARTH", et, lat, lon))

In [ ]:
def next_event(ets, elev, i_now, threshold=0.0):
    """Find the next time a body crosses the horizon, starting from index i_now."""

    # True where the body is above the threshold, False where it is below
    above = elev > threshold

    # Walk forward through time, one step at a time
    for i in range(i_now, len(elev) - 1):

        # If "above" changes between this step and the next, a crossing happened
        if above[i] != above[i + 1]:

            # How far between the two steps the crossing falls (0 to 1)
            # This is linear interpolation: assume a straight line between the two points
            frac = (threshold - elev[i]) / (elev[i + 1] - elev[i])

            # Convert that fraction into an actual time (in ephemeris seconds)
            t_cross = ets[i] + frac * (ets[i + 1] - ets[i])

            # If it was above before the crossing, the body is setting; otherwise rising
            kind = "sets" if above[i] else "rises"

            # Return the event type and the hours remaining from the starting point
            return kind, (t_cross - ets[i_now]) / 3600

    # No crossing found in the whole time range
    return None, None

In [ ]:
import pandas as pd
import numpy as np
import spiceypy as spice

# Ensure kernels are loaded
for k in ["naif0012.tls", "de440s.bsp", "pck00011.tpc", "moon_de440_220930.tf", "moon_pa_de440_200625.bpc"]:
    try:
        spice.furnsh(k)
    except:
        pass

# Define ets
et0 = spice.str2et("2026-11-14 00:00:00")
ets = et0 + np.arange(0, 30*86400, 3600)

# Choose the site (latitude south = negative, longitude east = positive)
lat, lon = -89.9, 0

# Compute the Sun and Earth elevation at every time step
sun = np.array([elevation("SUN", t, lat, lon) for t in ets])
earth = np.array([elevation("EARTH", t, lat, lon) for t in ets])

# Index 0 means "start from the first time step" (2026-11-14 00:00)
i_now = 0

# Sun: -0.27 degrees is the upper edge of the Sun's disk touching the horizon
kind, hrs = next_event(ets, sun, i_now, threshold=-0.27)
if kind:
    print("Sun", kind, "in", round(hrs, 1), "hours")
else:
    print("No Sun event in the 30 days")

# Earth: its disk is about 2 degrees wide, so use -1.0 as the threshold
kind, hrs = next_event(ets, earth, i_now, threshold=-1.0)
if kind:
    print("Earth", kind, "in", round(hrs, 1), "hours")
else:
    print("No Earth event in the 30 days")

In [ ]:
# Helper: find the longest unbroken stretch of True values in a True/False array
def longest_run(mask):
    best = 0      # longest stretch found so far
    current = 0   # length of the stretch we're currently in
    for value in mask:
        if value:
            current += 1                 # still in a stretch, so extend it
            best = max(best, current)    # remember it if it's the longest yet
        else:
            current = 0                  # stretch ended, so start over
    return best

# Time step is 1 hour (we built ets with a 3600-second step)
step_hours = 1

rows = []
for name, (la, lo) in sites.items():
    # Sun elevation for this site over the whole month
    s = np.array([elevation("SUN", t, la, lo) for t in ets])

    # Plot this site's Sun curve on the shared graph
    plt.plot(days, s, label=name)

    # Darkness = Sun fully below the horizon (using the -0.27 disk-edge threshold)
    dark = s < -0.27

    rows.append({
        "site": name,
        "sunlit_percent": round((~dark).mean() * 100, 1),            # % of time with sunlight
        "longest_dark_hours": longest_run(dark) * step_hours,         # worst night, in hours
        "longest_sunlit_hours": longest_run(~dark) * step_hours,      # longest continuous day
    })

# Finish the shared plot
plt.axhline(0, color="gray")
plt.xlabel("Days from 2026-11-14")
plt.ylabel("Sun elevation (deg)")
plt.legend(fontsize=7)
plt.grid(True)
plt.show()

# Show the summary table
pd.DataFrame(rows)

In [ ]:
# Range of Sun elevation over the 30 days (smallest and largest values)
print("Sun elevation:   min", round(sun.min(), 2), " max", round(sun.max(), 2))

# Same for Earth
print("Earth elevation: min", round(earth.min(), 2), " max", round(earth.max(), 2))

# Is the Sun above the disk-edge threshold at the start?
print("Sun up at start?", sun[0] > -0.27)

In [ ]:
# Two sites to compare: one near the south pole, one near the equator
compare = {
    "Near S. pole": (-89.9, 0),      # (latitude, longitude)
    "Apollo 11":    (0.67, 23.47),
}

for name, (la, lo) in compare.items():
    # Sun and Earth elevation at every hour of the month
    s = np.array([elevation("SUN", t, la, lo) for t in ets])
    e = np.array([elevation("EARTH", t, la, lo) for t in ets])

    # Print a short summary for this site
    print(name)
    print("  Sun elevation, lowest and highest:", round(s.min(), 1), round(s.max(), 1))
    print("  Sun is up for:  ", round((s > -0.27).mean() * 100, 1), "% of the month")
    print("  Earth is up for:", round((e > -1.0).mean() * 100, 1), "% of the month")

    # Draw this site's Sun curve on the same graph
    plt.plot(days, s, label=name)

# Finish the graph: horizon line, labels, legend
plt.axhline(0, color="gray")
plt.xlabel("Days from 2026-11-14")
plt.ylabel("Sun elevation (deg)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Helper: longest unbroken stretch of True values in a True/False array
def longest_run(mask):
    best = 0       # longest stretch found so far
    current = 0    # length of the stretch we're in right now
    for value in mask:
        if value:
            current += 1                  # stretch continues
            best = max(best, current)     # remember it if it's the longest yet
        else:
            current = 0                   # stretch ended, start over
    return best

step_hours = 1   # our time step is 1 hour

rows = []
for name, (la, lo) in sites.items():
    # Sun and Earth elevation at every hour of the month
    s = np.array([elevation("SUN", t, la, lo) for t in ets])
    e = np.array([elevation("EARTH", t, la, lo) for t in ets])

    # True where the Sun / Earth is visible (thresholds account for their disk size)
    sun_up = s > -0.27
    earth_up = e > -1.0

    rows.append({
        "site": name,
        "lat": la,
        "lon": lo,
        "sunlit_percent": round(sun_up.mean() * 100, 1),            # power: % of time with sunlight
        "longest_dark_hours": longest_run(~sun_up) * step_hours,     # power: worst night
        "earth_percent": round(earth_up.mean() * 100, 1),           # comms: % of time Earth is visible
        "longest_no_earth_hours": longest_run(~earth_up) * step_hours,  # comms: longest blackout
        "max_sun_elev": round(s.max(), 1),                          # best solar angle reached
    })

# Build the table and show it
results = pd.DataFrame(rows)
display(results)

# Save it as a file your partner can open
results.to_csv("site_results.csv", index=False)

In [ ]:
def az_el(body, et, lat_deg, lon_deg):
    """Return (azimuth, elevation) in degrees of a body as seen from a lunar site.
    Azimuth: compass direction clockwise from north (0 = north, 90 = east, 180 = south)."""

    # Convert the site's latitude and longitude to radians
    lat, lon = np.radians(lat_deg), np.radians(lon_deg)

    # Three directions at the site: straight up, due east, due north
    up = np.array([np.cos(lat)*np.cos(lon),
                   np.cos(lat)*np.sin(lon),
                   np.sin(lat)])
    east = np.array([-np.sin(lon), np.cos(lon), 0])
    north = np.array([-np.sin(lat)*np.cos(lon),
                      -np.sin(lat)*np.sin(lon),
                      np.cos(lat)])

    # Site position from the Moon's center
    site = R * up

    # Position of the Sun/Earth relative to the Moon, in the Moon-fixed frame
    pos, _ = spice.spkpos(body, et, FRAME, "NONE", "MOON")

    # Vector from the site to the body, shrunk to length 1
    v = np.array(pos) - site
    v = v / np.linalg.norm(v)

    # Elevation: angle above the horizon (same as before)
    el = np.degrees(np.arcsin(v @ up))

    # Azimuth: how far the body is toward east vs. north
    az = np.degrees(np.arctan2(v @ east, v @ north)) % 360   # % 360 keeps it between 0 and 360

    return az, el

# Try it: where is the Sun right now at the start time, near the south pole?
et = spice.str2et("2026-11-14 00:00:00")
az, el = az_el("SUN", et, -89.9, 0)
print("Sun azimuth:", round(az, 1), " elevation:", round(el, 2))

In [ ]:
# Fake terrain in ONE direction from the site:
# distance from the site (m) and height of the ground there (m, relative to the site)
distance = np.array([500, 1000, 2000, 5000, 10000])
height   = np.array([5,   30,   40,   300,  150])

# Angle from the site up to the top of each hill
angle = np.degrees(np.arctan(height / distance))

# The horizon in this direction is set by the hill that blocks the most sky
horizon = angle.max()
print("Hill angles:", angle.round(2))
print("Horizon angle:", round(horizon, 2), "degrees")

# Is the Sun visible? It must be higher than the horizon in that direction
sun_elev = 1.5
print("Sun visible?", sun_elev > horizon)

In [ ]:
# One horizon angle for every whole degree of azimuth (0 to 359)
# We make fake terrain: a flat 0.5 degree horizon with one big ridge between 80 and 100 degrees (east)
horizon_profile = np.full(360, 0.5)      # start with 0.5 degrees in every direction
horizon_profile[80:101] = 4.0            # a tall ridge to the east blocks up to 4 degrees

def horizon_at(az):
    """Look up the horizon angle for any azimuth by rounding to the nearest degree."""
    return horizon_profile[int(round(az)) % 360]    # % 360 wraps 360 back around to 0

# Check the Sun at every time step across the month, using the az_el function from before
lat, lon = -89.9, 0
visible = []                              # will hold True/False for each hour
for t in ets:
    az, el = az_el("SUN", t, lat, lon)   # where is the Sun right now?
    visible.append(el > horizon_at(az))   # visible only if it's higher than the horizon in that direction

visible = np.array(visible)               # turn the list into a numpy array

print("Flat-horizon model:    Sun up", round((sun > -0.27).mean() * 100, 1), "% of the month")
print("With fake ridge model: Sun up", round(visible.mean() * 100, 1), "% of the month")